# Technical note: Model structure and scoping

This note will briefly discuss Myokit's model structure and the scoping rules (which variable can access which).

It is **not** a replacement of the official syntax document: https://myokit.readthedocs.io/en/stable/syntax/model.html



## The basics

- A model has components.
- Components have variables.
- Variables can have further variables.

Variables fall into two groups, depending on their parent:

- Variables whose parent is a component are referred to as "component variables".
- Variables whose parent is a variable are referred to as "nested variables".

## Scope

Variables can access (depend on / use in their equations):

- Any component variables
- Any of their own children
- Any ancestor
- Any of their ancestor's children

To illustrate the rules for nested variables, we use the tree below

In [15]:
import myokit
m = myokit.Model()
q = m.add_component('q')
q.add_variable('t', rhs=0, binding='time')
a = q.add_variable('a', rhs=0)
b = a.add_variable('b', rhs=0)
c = a.add_variable('c', rhs=0)
d = b.add_variable('d', rhs=0)
e = b.add_variable('e', rhs=0)
f = c.add_variable('f', rhs=0)
g = d.add_variable('g', rhs=0)
h = f.add_variable('h', rhs=0)
vs = (a, b, c, d, e, f, g, h)

```
      a
     / \
    b   c
   / \   \ 
  d   e   f
 /         \
g           h
```

_Variables can access their own children, any ancestor, and any ancestor's children_

In [16]:
for v in vs:
    ok = []
    for w in vs:
        v.set_rhs(myokit.Name(w))
        try:
            m.validate()
        except myokit.IntegrityError:
            pass
        else:
            ok.append(w)
        v.set_rhs(0)
    print(f'{v.name()} can access ' + ', '.join(w.name() for w in ok))

a can access b, c
b can access a, c, d, e
c can access a, b, f
d can access a, b, c, e, g
e can access a, b, c, d
f can access a, b, c, h
g can access a, b, c, d, e
h can access a, b, c, f


- `a` can only access it's direct children
- `b` can access children `d` and `e`, ancestor `a`, and child-of-ancestor `c`
- `c` can access child `f`, ancestor `a`, child-of-ancestor `b`
- `d` can access child `g`, ancestors `a, b`, children-of-ancestors `c, e`
- `e` can access ancestors `a, b`, children-of-ancestors `c, d`
- `f` can access child `h`, ancestors `a, c`, child-of-ancestors `b`
- `g` can access ancestors `a, b, d`, children-of-ancestors `c, e`
- `h` can access ancestors `a, c, f`, children-of-ancestors `b`

Phrased the other way round,

Nested variables can be accessed by their descendants, their parent, or any of their parent's descendants.
```
      a
     / \
    b   c
   / \   \ 
  d   e   f
 /         \
g           h
```

In [17]:
for v in vs:
    ok = []
    for w in vs:
        w.set_rhs(myokit.Name(v))
        try:
            m.validate()
        except myokit.IntegrityError:
            pass
        else:
            ok.append(w)
        w.set_rhs(0)
    print(f'{v.name()} is accessible to ' + ', '.join(w.name() for w in ok))

a is accessible to b, c, d, e, f, g, h
b is accessible to a, c, d, e, f, g, h
c is accessible to a, b, d, e, f, g, h
d is accessible to b, e, g
e is accessible to b, d, g
f is accessible to c, h
g is accessible to d
h is accessible to f


- `a` is accessible to descendants `b, c, d, e, f, g, h`
- `b` is accessible to descendants `d, e, g`, parent `a`, and descendants-of-parent `c, f, h`
- `c` is accessible to descendants `f, h`, parent `a`, and descendants-of-parent `b, d, e, g`
- `d` is accessible to descendant `g`, parent `b`, and descendant-of-parent `e`
- `e` is accessible to parent `b`, and descendants-of-parent `d, g`
- `f` is accessible to descendant `h`, and parent `c`
- `g` is accessible to parent `d`
- `h` is accessible to parent `f`

## Access by name

- Variables in different components must be accessed as `component_name.variable_name`
- Variables in the same component, that are within scope according to the rules above, can be accessed by their name

In [18]:
for v in vs:
    ok = []
    for w in vs:
        if v == w:
            continue
        try:
            v.set_rhs(w.name())
        except myokit.ParseError:
            pass
        else:
            ok.append(w)
        v.set_rhs(0)
    print(f'{v.name()} can find ' + ', '.join(w.name() for w in ok))

a can find b, c
b can find a, c, d, e
c can find a, b, f
d can find a, b, c, e, g
e can find a, b, c, d
f can find a, b, c, h
g can find a, b, c, d, e
h can find a, b, c, f


## Uniqueness of names

- The same name may not appear twice in the same scope

```
      a
     / \
    b   c
   / \   \ 
  d   e   f
 /         \
g           h
```

In [25]:
for v in vs:
    ok = []
    old = v.name()
    for w in vs:
        if w == v:
            continue
        try:
            v.rename(w.name())
        except myokit.DuplicateName:
            pass
        else:
            ok.append(w)
    v.rename(old)
    if ok:
        print(f'{v.name()} can be renamed ' + ', '.join(w.name() for w in ok))
    else:
        print(f'{v.name()} can not be renamed')

a can not be renamed
b can not be renamed
c can not be renamed
d can be renamed f, h
e can be renamed f, h
f can be renamed d, e, g
g can be renamed f, h
h can be renamed d, e, g


- `a` is in everyone's scope
- `b` is in everyone's scope
- `c` is in everyone's scope
- `d` can access `a, b, c, e, g`, is accessible to `b, e, g`, leaving `f, h`
- `e` can access `a, b, c, d`, is accessible to `b, d, g`, leaving `f, h`
- `f` can access `a, b, c, h`, is accessible to `c, h`, leaving `d, e, g`
- `g` can access `a, b, c, d, e`, is accessible to `d`, leaving `f, h`
- `h` can access `a, b, c, f`, is accessible to `f`, leaving `d, e, g`
